# 🔄 Basic Agent Workflows wit Azure OpenAI (Responses API) (.NET)

## 📋 Workflow Orchestration Tutorial

Dis notebook dey show how to build correct **agent workflows** using Microsoft Agent Framework for .NET and Azure OpenAI (Responses API). You go learn how to create multi-step business process where AI agents dey work together to complete complex tasks through arranged orchestration patterns.

## 🎯 Wetin You Go Learn

### 🏗️ **Workflow Architecture Basics**
- **Workflow Builder**: Design and organise complex multi-step AI process dem
- **Agent Coordination**: Arrange many specialized agents inside workflows
- **Azure OpenAI (Responses API)**: Use Azure OpenAI Responses API for workflows
- **Visual Workflow Design**: Create and see workflow structures well well to understand dem better

### 🔄 **Process Orchestration Patterns**
- **Sequential Processing**: Arrange many agent tasks one after another for correct order
- **State Management**: Keep context and data movement for all steps of workflow
- **Error Handling**: Do strong error recovery and make workflow steady
- **Performance Optimization**: Design workflows wey dey make work fast for enterprise-level business

### 🏢 **Enterprise Workflow Practice Dem**
- **Business Process Automation**: Make complex company workflow run automatically
- **Content Production Pipeline**: Editorial workflows wey get review and approval steps
- **Customer Service Automation**: Multi-step solution for customer inquiry
- **Data Processing Workflows**: ETL workflows with AI-powered data change

## ⚙️ Wetin You Need & How To Set

### 📦 **NuGet Packages We Need**

Dis workflow demo dey use some important .NET packages:

```xml
<!-- Core AI Framework -->
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.0" />

<!-- Azure OpenAI (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />
<PackageReference Include="Azure.Identity" Version="1.13.1" />

<!-- Agent Framework (Local Development) -->
<!-- Microsoft.Agents.AI.dll - Core agent abstractions -->
<!-- Microsoft.Agents.AI.OpenAI.dll - Azure OpenAI (Responses API) integration -->

<!-- Configuration and Environment -->
<PackageReference Include="DotNetEnv" Version="3.1.1" />
```

### 🔑 **Azure OpenAI Setup**

**Environment Setup (.env file):**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-5-mini
```

**How to Access Azure OpenAI:**
1. Create Azure OpenAI resource for Azure portal
2. Deploy model (for example, `gpt-5-mini`) and remember deployment name
3. Sign in with `az login` and set environment variables as e show for above

### 🏗️ **Workflow Architecture Overview**

```mermaid
graph TD
    A[Workflow Bɔ́lụ] --> B[Agent Registri]
    B --> C[Workflow Execution Engine]
    C --> D[Agent 1: Content Generator
    C --> E[Agent 2: Content Reviewer 
    D --> F[Workflow Results]
    E --> F
    G[Azure OpenAI (Responses API)] --> D
    G --> E
```

**Main Parts:**
- **WorkflowBuilder**: Main engine wey dey design workflows
- **AIAgent**: Individual specialized agents wey get special skills
- **Azure OpenAI Client**: Azure OpenAI Responses API join
- **Execution Context**: Manages state and data flow between different workflow steps

## 🎨 **Enterprise Workflow Design Patterns**

### 📝 **Content Production Workflow**
```
User Request → Content Generation → Quality Review → Final Output
```

### 🔍 **Document Processing Pipeline**
```
Document Input → Analysis → Extraction → Validation → Structured Output
```

### 💼 **Business Intelligence Workflow**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

### 🤝 **Customer Service Automation**
```
Customer Inquiry → Classification → Processing → Response Generation → Follow-up
```

## 🏢 **Enterprise Benefits**

### 🎯 **Reliability & Scalability**
- **Deterministic Execution**: Same, repeatable workflow result always
- **Error Recovery**: Soft handling of failure anytime for workflow step
- **Performance Monitoring**: Watch execution metrics and chances for improvement
- **Resource Management**: Use AI model resources like beta

### 🔒 **Security & Compliance**
- **Secure Authentication**: Microsoft Entra ID authentication wit `az login` (AzureCliCredential)
- **Audit Trails**: Complete log of workflow execution and decisions
- **Access Control**: Correct permission for workflow running and watching
- **Data Privacy**: Keep sensitive info safe all through workflows

### 📊 **Observability & Management**
- **Visual Workflow Design**: Clear way to show process flows and connections
- **Execution Monitoring**: Watch workflow progress and how e dey perform for real time
- **Error Reporting**: Deep error analysis and debugging skill
- **Performance Analytics**: Metrics for improvement and capacity planning

Make we build your first correct enterprise-ready AI workflow! 🚀


In [1]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

Installed Packages Microsoft.Extensions.AI, 9.9.1

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

Installed Packages System.ClientModel, 1.6.1

In [3]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"
#r "nuget: OpenTelemetry.Api, 1.0.0"

Installed Packages Azure.Identity, 1.15.0 OpenTelemetry.Api, 1.0.1 System.Linq.Async, 6.0.3

In [5]:

#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3

In [ ]:

#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.2

In [7]:
#r "nuget: DotNetEnv, 3.1.1"

Installed Packages DotNetEnv, 3.1.1

In [8]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 9.9.0-preview.1.25458.4"

In [ ]:
using System;
using System.ComponentModel;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;

In [10]:
 using DotNetEnv;

In [11]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-5-mini";

In [ ]:
// The Azure OpenAI client is created directly from the endpoint and Azure CLI credential — no custom client options are required.

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [15]:
const string ReviewerAgentName = "Concierge";
const string ReviewerAgentInstructions = @"
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. ";

In [16]:
const string FrontDeskAgentName = "FrontDesk";
const string FrontDeskAgentInstructions = @"""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """;

In [ ]:
AIAgent reviewerAgent = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:ReviewerAgentName,instructions:ReviewerAgentInstructions);
AIAgent frontDeskAgent  = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:FrontDeskAgentName,instructions:FrontDeskAgentInstructions);

In [18]:
var workflow = new WorkflowBuilder(frontDeskAgent)
            .AddEdge(frontDeskAgent, reviewerAgent)
            .Build();

In [19]:
ChatMessage userMessage = new ChatMessage(ChatRole.User, [
	new TextContent("I would like to go to Paris.") 
]);

In [20]:
StreamingRun run = await InProcessExecution.StreamAsync(workflow, userMessage);

In [21]:
await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
string id="";
string messageData="";
await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
{
    if (evt is AgentRunUpdateEvent executorComplete)
    {
        if(id=="")
        {
            id=executorComplete.ExecutorId;
        }
        if(id==executorComplete.ExecutorId)
        {
            messageData+=executorComplete.Data.ToString();
        }
        else
        {
            id=executorComplete.ExecutorId;
        }
        // Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}

Console.WriteLine(messageData);

Visit the Louvre Museum. It's a must-see for art enthusiasts and history lovers.That recommendation is quite popular and likely to attract many tourists. To refine it for a more local and authentic experience, consider suggesting an alternative that focuses on smaller, lesser-known art venues or galleries. Look for places where local artists exhibit or community spaces that host cultural events. This approach allows travelers to connect with the local art scene more intimately, away from the typical tourist routes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
